# 05 - Hybrid Modeling

Combine ML and DL models to reduce bias-variance and improve robustness.


## Why Hybrid Models

**Definition**  
Hybrid models blend complementary inductive biases from different model families.

**Theory**  
Linear models capture simple trend; trees capture nonlinear interactions; sequence models capture memory. Ensemble aggregation can reduce variance.

**Mathematical Intuition**  
Weighted combination: `ŷ = Σ w_i ŷ_i`, where `w_i` learned from validation objective.

**Financial Intuition**  
Market dynamics change by regime; no single model dominates all periods.

**Business Impact**  
Hybrid strategy often yields more stable risk-adjusted error profile.

**Real-World Example**  
During volatility shocks, one submodel may fail while others remain calibrated.

**Visual Explanation**  
Code cells below generate real plots from Apple market data.

**Code Explanation**  
Each code block is annotated inline and uses shared production modules from `src/`.

**Interpretation of Results**  
After each output, interpret what signal quality, risk, and forecasting reliability imply.


In [ ]:

import sys
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.forecast_pipeline import ForecastingFramework
from src.data_loader import load_stock_data, split_data
from src.features import create_features
from src.evaluation import regression_metrics
from src.visualization import *

OUT = PROJECT_ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
FAST_NOTEBOOK_MODE = os.getenv('NOTEBOOK_FULL_RUN', '0') != '1'

def make_framework():
    framework = ForecastingFramework(str(CONFIG_PATH))
    if FAST_NOTEBOOK_MODE:
        framework.config['models']['automl']['lazypredict'] = False
        framework.config['models']['automl']['pycaret'] = False
        framework.config['models']['automl']['flaml'] = False
        framework.config['models']['deep_learning']['epochs'] = 8
        framework.config['models']['deep_learning']['early_stopping_patience'] = 3
        framework.config['weight_optimization']['methods'] = ['grid']
    return framework

framework = make_framework()


In [ ]:

framework = make_framework()
framework.load_data()

hybrid_out = framework.train_hybrids(horizon=1)
display(hybrid_out['leaderboard'])
display(hybrid_out['val_leaderboard'])

hybrid_out['leaderboard'].to_csv(OUT / 'tables/05_hybrid_h1.csv', index=False)
hybrid_out['val_leaderboard'].to_csv(OUT / 'tables/05_hybrid_h1_validation.csv', index=False)
